In [ ]:
# add the link to your repo here
!git clone #repo-link

%cd #repo-name

!ls

In [ ]:
from google.colab import userdata
import os

# add secrets named OPENAI_API_KEY and GOOGLE_API_KEY
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!cp -r #/google/drive/path/to/dev/train/and/test/data

# The Algorithmic councils of THM SimpleText 2026

## First Council - BERT Classifiers

### The 5 models

1. DistilBERT: lighter bert model, fast and small.

2. RoBERTa: used by AIIRLab, for multi-label classification

3. SciBERT: scientifc bert model

4. BERT-base: industry standard

In [ ]:
# cache clearing
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='urllib3')
import gc
import torch
gc.collect()

# 1. Generating BERT data

Set `train_size` and `eval_size` according to the GPUs capabilities.

## Don't forget to adjust the name of the paths to your configuration

In [ ]:
from berthold_classifying import BERTCouncil
from stripper import strip_confident_nones

distilbert = BERTCouncil(distilbert=True)
if not os.path.exists("/content/drive/MyDrive/colab_data/submission_distilbert_sentences_DEV.json"):
    status = distilbert.run_all_models(
        train_path="Task_2_new/train_data.json",
        test_path="Task_2_new/dev_data.json",
        train_size=16,
        eval_size=32
    )
    print(status)
    os.rename(
            "submission_distilbert_sentences.json",
            "submission_distilbert_sentences_DEV.json"
    )
    !cp ./submission_distilbert_sentences_DEV.json #/google/drive/path/to/data

if not os.path.exists("/content/drive/MyDrive/colab_data/submission_distilbert_sentences.json"):
    status = distilbert.run_all_models(
        train_path="Task_2_new/train_data.json",
        test_path="Task_2_new/test_data.json",
        train_size=16,
        eval_size=32
    )
    print(status)
    !cp ./submission_distilbert_sentences.json #/google/drive/path/to/data

if not os.path.exists("/content/drive/MyDrive/colab_data/Task_2_new/test_data_filtered.json"):
    strip_confident_nones(
        original_test_path="Task_2_new/test_data.json",
        distilbert_predictions_path="submission_distilbert_sentences.json",
        output_filtered_path="test_data_filtered.json",
        none_threshold=0.8
    )
    !cp ./Task_2_new/test_data_filtered.json #/google/drive/path/to/dev/train/and/test/data

council = BERTCouncil(roberta=True)
if not os.path.exists("/content/drive/MyDrive/colab_data/submission_roberta_sentences_DEV.json"):
    status = council.run_all_models(
        train_path="Task_2_new/train_data.json",
        test_path="Task_2_new/dev_data.json",
        train_size=16,
        eval_size=32
    )
    print(status)
    os.rename(
            "submission_roberta_sentences.json",
            "submission_roberta_sentences_DEV.json"
    )
    !cp ./submission_roberta_sentences_DEV.json #/google/drive/path/to/data

if not os.path.exists("/content/drive/MyDrive/colab_data/submission_roberta_sentences.json"):
    status = council.run_all_models(
        train_path="Task_2_new/train_data.json",
        test_path="Task_2_new/test_data_filtered.json",
        train_size=16,
        eval_size=32
    )
    print(status)
    !cp ./submission_roberta_sentences.json #/google/drive/path/to/data

council = BERTCouncil(scibert=True)
if not os.path.exists("/content/drive/MyDrive/colab_data/submission_scibert_sentences_DEV.json"):
    status = council.run_all_models(
        train_path="Task_2_new/train_data.json",
        test_path="Task_2_new/dev_data.json",
        train_size=16,
        eval_size=32
    )
    print(status)
    os.rename(
            "submission_scibert_sentences.json",
            "submission_scibert_sentences_DEV.json"
    )
    !cp ./submission_scibert_sentences_DEV.json #/google/drive/path/to/data

if not os.path.exists("/content/drive/MyDrive/colab_data/submission_scibert_sentences.json"):
    status = council.run_all_models(
        train_path="Task_2_new/train_data.json",
        test_path="Task_2_new/test_data_filtered.json",
        train_size=16,
        eval_size=32
    )
    print(status)
    !cp ./submission_scibert_sentences.json #/google/drive/path/to/data

council = BERTCouncil(bert=True)
if not os.path.exists("/content/drive/MyDrive/colab_data/submission_bert_sentences_DEV.json"):
    council = BERTCouncil(bert=True)
    status = council.run_all_models(
        train_path="Task_2_new/train_data.json",
        test_path="Task_2_new/dev_data.json",
        train_size=16,
        eval_size=32
    )
    print(status)
    os.rename(
            "submission_bert_sentences.json",
            "submission_bert_sentences_DEV.json"
    )
    !cp ./submission_bert_sentences_DEV.json #/google/drive/path/to/data

if not os.path.exists("/content/drive/MyDrive/colab_data/submission_bert_sentences.json"):
    status = council.run_all_models(
        train_path="Task_2_new/train_data.json",
        test_path="Task_2_new/test_data_filtered.json",
        train_size=16,
        eval_size=32
    )
    print(status)
    !cp ./submission_bert_sentences.json #/google/drive/path/to/data

## 2. The first judge
Random Forest part 1. BERT hunger games

`len` acts as a filter to check for smaller sentences (which are more likely to be "none") or larger sentences, which imply errors

`ratio` is a small classification, if the ratio is 1.5, an overgeneration is likely, if the ratio is above 3.0 generation failure or repetitive content is likely

`tiny` counters false positives

In [ ]:
from i_know_where_you_stand import (create_meta_ensemble_dev, run_inference_and_save)

model_files = [
    "submission_distilbert_sentences_DEV.json",
    "submission_bert_sentences_DEV.json",
    "submission_roberta_sentences_DEV.json",
    "submission_scibert_sentences_DEV.json",
]

meta_model, df_train_meta, feature_cols = create_meta_ensemble_dev(
    model_files=model_files,
    dev_json_path="Task_2_new/dev_data.json",
)

feature_cols = [col.replace("_DEV", "") for col in feature_cols]

test_model_files = [
    "submission_distilbert_sentences.json",
    "submission_bert_sentences.json",
    "submission_roberta_sentences.json",
    "submission_scibert_sentences.json",
]

run_inference_and_save(
    meta_model=meta_model,
    test_model_files=test_model_files,
    test_json_path="Task_2_new/test_data.json",
    feature_cols=feature_cols,
    output_file="submission_meta_ensemble.json",
    team_name="TeamTEAM",
)

files.download("submission_meta_ensemble.json")
files.download("submission_meta_ensemble_docs.json")

# 3. Stripping the meta ensemble output
this one yoinks all "none" entries and entries, where each bert model agrees on one error

In [ ]:
import json
import pandas as pd

NONE_PROB_THRESHOLD = 0.7       # sentence is confidently none
BERT_AGREEMENT_THRESHOLD = 0.5  # every model must agree

NON_NONE_LABELS = [
    "GENERATION_FAILURE",
    "LEAKED_INSTRUCTIONS",
    "UNGROUNDED_INJECTION",
    "REPETITIVE_CONTENT",
    "GROUNDED_OVERGENERATION",
]

with open("submission_meta_ensemble.json", "r") as f:
    ensemble = json.load(f)

print(f"Original record count: {len(ensemble)}")

def is_confident_none(record):
    return record.get("None_prob", 0.0) >= NONE_PROB_THRESHOLD

after_pass1 = [r for r in ensemble if not is_confident_none(r)]
removed_pass1 = len(ensemble) - len(after_pass1)
print(f"Pass 1 removed (confident None ≥ {NONE_PROB_THRESHOLD}): {removed_pass1}")
print(f"Remaining after pass 1: {len(after_pass1)}")


def all_models_agree_on_error(record):
    bert_keys = [k for k in record if k.startswith("bert_")]

    if not bert_keys:
        return False

    for error_label in NON_NONE_LABELS:
        all_agree = all(
            record[bert_model].get(f"{error_label}_prob", 0.0) > BERT_AGREEMENT_THRESHOLD
            for bert_model in bert_keys
        )
        if all_agree:
            return True
    return False


after_pass2 = [r for r in after_pass1 if not all_models_agree_on_error(r)]
removed_pass2 = len(after_pass1) - len(after_pass2)
print(f"Pass 2 removed (all BERT models agree on one error class): {removed_pass2}")
print(f"Remaining after pass 2: {len(after_pass2)}")

output_path = "submission_meta_ensemble_stripped.json"
with open(output_path, "w") as f:
    json.dump(after_pass2, f, indent=4)
files.download(output_path)

print(f"\nSaved {len(after_pass2)} records → {output_path}")
if len(after_pass2) == 0:
    print(
        "\nNote: stripped file is empty — expected at sample_size=100 since "
        "all BERT models predict None for every sentence. Re-run with a larger "
        "sample size to get meaningful uncertain sentences for LLM review."
    )
else:
    labels_remaining = [r["predicted_label"] for r in after_pass2]
    label_counts = pd.Series(labels_remaining).value_counts()
    print("\nLabel distribution in stripped file:")
    print(label_counts.to_string())

# 4. The second council
chatgpt uses his insane authority as the only judge, jury and executioner to decide the fate of the the dataset

In [ ]:
import pandas as pd
import json
from llm_council import run_council_batch

with open("submission_meta_ensemble_stripped.json", "r") as f:
    stripped_list = json.load(f)

prompts_list = {
    "Factuality_Check": (
        "Look at this source: '{src}'. "
        "Does the simplification '{simp}' hallucinate new facts? Answer YES or NO."
    ),
    "Error_Detection": (
        "Analyze the simplification '{simp}' against the source '{src}'.\n"
        "Identify errors from this taxonomy:\n"
        "- GENERATION_FAILURE\n"
        "- LEAKED_INSTRUCTIONS\n"
        "- UNGROUNDED_INJECTION\n"
        "- REPETITIVE_CONTENT\n"
        "- GROUNDED_OVERGENERATION\n"
        "If clean, state 'NO ERRORS'."
    ),
}

sample_size = 50 # change to none, for final run (note that sample size is based on meta_ensemble_stripped)
entries_to_run = stripped_list[:sample_size] if sample_size else stripped_list

print(f"Sending {len(entries_to_run)} sentences to LLM council (batched)...")

all_council_results = await run_council_batch(
    prompts_list,
    entries_to_run,
    concurrency=5, # the parallel batches
)

print(f"Done. {len(all_council_results)} LLM responses collected.")
df_council = pd.DataFrame(all_council_results)
df_council.head()

# 5. Merging back together
takes llm outputs and puts them back to the former meta_ensemble (full)


In [ ]:
from collections import defaultdict

with open("submission_meta_ensemble.json", "r") as f:
    full_ensemble = json.load(f)

llm_lookup = defaultdict(list)
for row in all_council_results:
    llm_lookup[row["snt_id"]].append(row)

PLACEHOLDER = "empty, due to none"

task_model_pairs = sorted({
    (r["task"], r["model"]) for r in all_council_results
})

def build_llm_columns(snt_id):
    responses = llm_lookup.get(snt_id, [])

    if not responses:
        cols = {
            f"llm_{model.lower()}_{task.lower()}": PLACEHOLDER
            for task, model in task_model_pairs
        }
        cols["llm_reviewed"] = False
    else:
        cols = {
            f"llm_{r['model'].lower()}_{r['task'].lower()}": r["response"]
            for r in responses
        }
        cols["llm_reviewed"] = True

    return cols

merged_ensemble = []
for record in full_ensemble:
    llm_cols = build_llm_columns(record["snt_id"])
    merged_ensemble.append({**record, **llm_cols})

output_json = "submission_meta_ensemble_with_llm.json"
with open(output_json, "w") as f:
    json.dump(merged_ensemble, f, indent=4)
files.download(output_json)

df_merged = pd.DataFrame(merged_ensemble)

reviewed = int(df_merged["llm_reviewed"].sum())
skipped  = int((~df_merged["llm_reviewed"]).sum())

print(f"\nMerged ensemble saved.")
print(f"  Total sentences : {len(df_merged)}")
print(f"  LLM reviewed    : {reviewed}")
print(f"  Skipped (None)  : {skipped}")
print(f"  JSON → {output_json}")

df_merged.head()

### short intermission
just some small adjustments that combines these results
important for later

In [ ]:
from auditor import AlignmentAuditor

auditor = AlignmentAuditor("submission_meta_ensemble_with_llm.json")
final_report = auditor.audit()

high_risk = final_report[final_report["final_decision"] == "REVIEW REQUIRED"] #best case is manuel review

if not high_risk.empty:
    llm_cols = [c for c in final_report.columns if c.startswith("llm_") and c != "llm_reviewed"]
    display(
        high_risk[[
            "snt_id",
            "source_sentence",
            "simplified_sentence",
            "predicted_label",
            "conflict_score",
        ] + llm_cols]
    )
else:
    print("No sentences flagged for review.")

skipped = final_report[final_report["final_decision"] == "SKIPPED"]
print(f"\n{len(skipped)} sentences skipped (stripped before LLM review).")

final_report.to_json("audited_meta_ensemble.json", orient="records", indent=4)

# 6. Last judge
ALL IN ONE EXCLUSIVE HOLY SH*T (i just got too lazy to create a script for that)
translates llm results into random forest readable ones and zeros (everybody listening?)
then the random forest decides the final verdict

In [ ]:
import re
import asyncio
import pandas as pd
import json
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from llm_council import LLMCouncil

with open("audited_meta_ensemble.json", "r") as f:
    audited_list = json.load(f)
audited_data = pd.DataFrame(audited_list)

reviewed_df = audited_data[audited_data["final_decision"] != "SKIPPED"].copy()
skipped_df  = audited_data[audited_data["final_decision"] == "SKIPPED"].copy()
print(f"Reviewed: {len(reviewed_df)} | Skipped: {len(skipped_df)}")


def extract_llm_signals(row):
    llm_cols = [c for c in audited_data.columns
                if c.startswith("llm_") and c != "llm_reviewed"]

    text = " ".join(
        str(row[c]) for c in llm_cols
        if row.get(c) and row[c] != "empty, due to none"
    ).lower()

    return {
        "llm_hallucination_detected": 1 if any(
            k in text for k in ["injection", "hallucinate", "ungrounded"]
        ) else 0,
        "llm_instruction_leak": 1 if any(
            k in text for k in ["assistant", "instruction", "leaked"]
        ) else 0,
        "llm_style_score": 1 if ("perfect" in text or "good" in text) else 0,
        "llm_conflict_intensity": float(row.get("conflict_score", 0.0)),
    }

llm_features = reviewed_df.apply(
    lambda x: pd.Series(extract_llm_signals(x)), axis=1
)
ultimate_df = pd.concat([reviewed_df, llm_features], axis=1)


gemini_cols  = [c for c in audited_data.columns
                if c.startswith("llm_gemini_") and c != "llm_reviewed"]
chatgpt_cols = [c for c in audited_data.columns
                if c.startswith("llm_chatgpt_") and c != "llm_reviewed"]

def join_responses(row, cols):
    return " | ".join(
        str(row[c]) for c in cols
        if row.get(c) and row[c] != "empty, due to none"
    )

judge_input = ultimate_df[
    ["snt_id", "source_sentence", "simplified_sentence"]
].copy()
judge_input["model_a_all"] = reviewed_df.apply(
    lambda r: join_responses(r, gemini_cols), axis=1
)
judge_input["model_b_all"] = reviewed_df.apply(
    lambda r: join_responses(r, chatgpt_cols), axis=1
)

async def run_all_judges(judge_input_df):
    council_instance = LLMCouncil()

    async def judge_one(row):
        judge_prompt = f"""
        You are the Supreme Judge of a Text Simplification Council.
        SOURCE: "{row['source_sentence']}"
        SIMPLIFICATION: "{row['simplified_sentence']}"

        CRITIQUE A (Gemini): {row['model_a_all']}
        CRITIQUE B (ChatGPT): {row['model_b_all']}

        TASK:
        1. Evaluate which critique was more accurate based on the source.
        2. Output a trust score from 0.0 to 1.0 for each model.

        FORMAT:
        TRUST_MODEL_A: [score]
        TRUST_MODEL_B: [score]
        WINNER: [Model A or Model B]
        """
        g_verdict, c_verdict = await asyncio.gather(
            council_instance._ask_gemini(judge_prompt),
            council_instance._ask_chatgpt(judge_prompt),
        )
        return row["snt_id"], g_verdict, c_verdict

    tasks = [judge_one(row) for _, row in judge_input_df.iterrows()]
    return await asyncio.gather(*tasks)


def parse_trust_scores(text):
    ta, tb = 0.5, 0.5
    try:
        ta_m = re.search(r"TRUST_MODEL_A:\s*([\d\.]+)", text)
        tb_m = re.search(r"TRUST_MODEL_B:\s*([\d\.]+)", text)
        if ta_m: ta = float(ta_m.group(1))
        if tb_m: tb = float(tb_m.group(1))
    except Exception:
        pass
    return ta, tb


print(f"Running Supreme Judge on {len(judge_input)} sentences...")
raw_verdicts = await run_all_judges(judge_input)

judge_results = []
for snt_id, g_verdict, c_verdict in raw_verdicts:
    g_ta, g_tb = parse_trust_scores(g_verdict)
    c_ta, c_tb = parse_trust_scores(c_verdict)
    judge_results.append({
        "snt_id": snt_id,
        "judge_trust_model_a": (g_ta + c_ta) / 2,
        "judge_trust_model_b": (g_tb + c_tb) / 2,
        "judge_winner_encoded": 1 if (g_ta + c_ta) > (g_tb + c_tb) else 0,
        "judge_disagreement": abs(g_ta - c_ta),
    })

df_judge = pd.DataFrame(judge_results)
ultimate_df = pd.merge(ultimate_df, df_judge, on="snt_id", how="left")


NON_NONE_PROB_COLS = [
    "GENERATION_FAILURE_prob",
    "LEAKED_INSTRUCTIONS_prob",
    "UNGROUNDED_INJECTION_prob",
    "REPETITIVE_CONTENT_prob",
    "GROUNDED_OVERGENERATION_prob",
]
LLM_COLS = [
    "llm_hallucination_detected",
    "llm_instruction_leak",
    "llm_conflict_intensity",
]
JUDGE_COLS = [
    "judge_trust_model_a",
    "judge_trust_model_b",
    "judge_winner_encoded",
    "judge_disagreement",
]

feature_cols = NON_NONE_PROB_COLS + LLM_COLS + JUDGE_COLS

for col in feature_cols:
    if col not in ultimate_df.columns:
        ultimate_df[col] = 0.0

X = ultimate_df[feature_cols].fillna(0.0)
y = (ultimate_df["final_decision"] == "REVIEW REQUIRED").astype(int)


if y.nunique() > 1:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
else:
    print("Warning: only one class present in reviewed set — skipping stratify")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

rf_supreme = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_supreme.fit(X_train, y_train)

y_pred = rf_supreme.predict(X_test)
print("\nSupreme RF — Test Set Report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["VALIDATED", "REVIEW REQUIRED"],
    zero_division=0,
))

probs = rf_supreme.predict_proba(X)
ultimate_df["supreme_confidence_score"] = (
    probs[:, 1] if probs.shape[1] > 1 else 1.0
)


for col in LLM_COLS + JUDGE_COLS + ["supreme_confidence_score"]:
    skipped_df[col] = 0.0

final_df = pd.concat([ultimate_df, skipped_df], ignore_index=True)
final_df = final_df.sort_values("snt_id").reset_index(drop=True)

final_df.to_json("ultimate_council_results.json", orient="records", indent=4)
print(f"  Reviewed + scored : {len(ultimate_df)}")
print(f"  Skipped (clean)   : {len(skipped_df)}")

epic graphic (useless unless sample size above 100)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

importances = rf_supreme.feature_importances_
feat_importances = pd.Series(importances, index=feature_cols).sort_values(ascending=True)

def get_color(col):
    if "llm" in col:
        return "#e67e22"
    if "judge" in col:
        return "#9b59b6"
    return "#3498db"

colors = [get_color(col) for col in feat_importances.index]

fig, ax = plt.subplots(figsize=(11, 7))
feat_importances.plot(kind="barh", color=colors, ax=ax, edgecolor="white", linewidth=0.5)

ax.set_title(
    "Supreme Judge: Feature Importance (BERT vs. LLM vs. Judge)",
    fontsize=14, fontweight="bold", pad=14
)
ax.set_xlabel("Importance Score (Influence on Final Decision)", fontsize=12)
ax.set_ylabel("Feature", fontsize=12)
ax.grid(axis="x", linestyle="--", alpha=0.6)
ax.spines[["top", "right"]].set_visible(False)

legend_elements = [
    Line2D([0], [0], color="#3498db", lw=5, label="BERT Ensemble probabilities"),
    Line2D([0], [0], color="#e67e22", lw=5, label="LLM Council signals"),
    Line2D([0], [0], color="#9b59b6", lw=5, label="Supreme Judge scores"),
]
ax.legend(handles=legend_elements, loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
import json
import zipfile
import os
import pandas as pd
from collections import defaultdict

TEAM_NAME   = "THM"
METHOD_USED = "UltimateCouncil"
RUN_ID_22   = f"{TEAM_NAME}_Task22b_{METHOD_USED}"
RUN_ID_21   = f"{TEAM_NAME}_Task21b_{METHOD_USED}"

VALID_LABELS = {
    "None",
    "GENERATION_FAILURE",
    "LEAKED_INSTRUCTIONS",
    "UNGROUNDED_INJECTION",
    "REPETITIVE_CONTENT",
    "GROUNDED_OVERGENERATION",
}

with open("ultimate_council_results.json", "r") as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(f"Loaded {len(df)} sentence records across {df['doc_id'].nunique()} documents.")


def resolve_label(row):
    if float(row.get("supreme_confidence_score", 0.0)) > 0.0:
        if row.get("final_decision") == "REVIEW REQUIRED":
            label = row.get("predicted_label", "None")
            return label if label in VALID_LABELS else "None"
        else:
            return "None"

    if row.get("llm_reviewed", False):
        label = row.get("predicted_label", "None")
        return label if label in VALID_LABELS else "None"

    return "None"


df["final_label"] = df.apply(resolve_label, axis=1)

invalid = df[~df["final_label"].isin(VALID_LABELS)]
if not invalid.empty:
    print(f"WARNING: {len(invalid)} invalid labels found — forcing to 'None'")
    df.loc[~df["final_label"].isin(VALID_LABELS), "final_label"] = "None"


def extract_sentence_index(snt_id):
    try:
        return int(snt_id.rsplit("_", 1)[-1])
    except (ValueError, IndexError):
        return 0

df["snt_index"] = df["snt_id"].apply(extract_sentence_index)
df = df.sort_values(["doc_id", "snt_index"])

doc_groups = df.groupby("doc_id", sort=False)


submission_22 = []
for doc_id, group in doc_groups:
    submission_22.append({
        "id": doc_id,
        "labels": group["final_label"].tolist(),
        "run_id": RUN_ID_22,
    })

output_22 = f"{RUN_ID_22}.json"
with open(output_22, "w") as f:
    json.dump(submission_22, f, indent=2)
files.download(output_22)
print(f"Task 2.2 → {output_22}  ({len(submission_22)} documents)")


def to_binary(label):
    return "None" if label == "None" else "Overgeneration"

submission_21 = []
for doc_id, group in doc_groups:
    submission_21.append({
        "id": doc_id,
        "labels": [to_binary(l) for l in group["final_label"].tolist()],
        "run_id": RUN_ID_21,
    })

output_21 = f"{RUN_ID_21}.json"
with open(output_21, "w") as f:
    json.dump(submission_21, f, indent=2)
files.download(output_21)
print(f"Task 2.1 → {output_21}  ({len(submission_21)} documents)")


zip_22 = f"{RUN_ID_22}.zip"
zip_21 = f"{RUN_ID_21}.zip"

with zipfile.ZipFile(zip_22, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(output_22)

with zipfile.ZipFile(zip_21, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(output_21)

ids_22 = [r["id"] for r in submission_22]
assert len(ids_22) == len(set(ids_22)), "Duplicate doc IDs found!"

mismatches = []
for record in submission_22:
    doc_id = record["id"]
    expected = len(df[df["doc_id"] == doc_id])
    actual   = len(record["labels"])
    if expected != actual:
        mismatches.append((doc_id, expected, actual))
if mismatches:
    print(f"WARNING: {len(mismatches)} label count mismatches:")
    for doc_id, exp, act in mismatches[:5]:
        print(f"  {doc_id}: expected {exp}, got {act}")
else:
    print("Label counts match sentence counts for all documents")

all_labels_22 = {l for r in submission_22 for l in r["labels"]}
binary_labels  = {"None", "Overgeneration"}
mixed = all_labels_22 - VALID_LABELS
if mixed:
    print(f"WARNING: unexpected labels in Task 2.2 output: {mixed}")
else:
    print("All Task 2.2 labels are valid taxonomy labels")

label_counts = df["final_label"].value_counts()
doc_positive  = sum(1 for r in submission_22 if any(l != "None" for l in r["labels"]))
print(label_counts.to_string())
print(f"\nDocuments with at least one overgeneration: {doc_positive}/{len(submission_22)}")